In [105]:
import json

In [106]:
import sys

sys.path.append("/local/home/amamun/projects/RAGent")

In [107]:
path_aware_result = "/local/home/amamun/projects/RAGent/localization_results/retrieval_results_code_test_with_path_k50_20lines_with_patch_file.json"
normal_result = "/local/home/amamun/projects/RAGent/localization_results/rag_results_code_splitter.json"

In [108]:
with open(path_aware_result, "r") as f:
    path_aware_data = json.load(f)

with open(normal_result, "r") as f:
    normal_data = json.load(f)

In [109]:
exclusively_path_aware = []
how_they_failed = []

for path_aware, normal in zip(path_aware_data, normal_data):
    assert path_aware["swe_data_index"] == normal["swe_data_index"]
    path_aware_retrieved_files = path_aware["retrieved_files"]
    normal_retrieved_files = normal["retrieved_files"]
    patch_file = path_aware["patch_file"]

    if patch_file in path_aware_retrieved_files[:10] and patch_file not in normal_retrieved_files[:10]:
        exclusively_path_aware.append(path_aware)
        how_they_failed.append(normal)


In [110]:
len(exclusively_path_aware)

43

In [111]:
how_they_failed[0]

{'swe_data_index': 0,
 'problem_statement': "Modeling's `separability_matrix` does not compute separability correctly for nested CompoundModels\nConsider the following model:\r\n\r\n```python\r\nfrom astropy.modeling import models as m\r\nfrom astropy.modeling.separable import separability_matrix\r\n\r\ncm = m.Linear1D(10) & m.Linear1D(5)\r\n```\r\n\r\nIt's separability matrix as you might expect is a diagonal:\r\n\r\n```python\r\n>>> separability_matrix(cm)\r\narray([[ True, False],\r\n       [False,  True]])\r\n```\r\n\r\nIf I make the model more complex:\r\n```python\r\n>>> separability_matrix(m.Pix2Sky_TAN() & m.Linear1D(10) & m.Linear1D(5))\r\narray([[ True,  True, False, False],\r\n       [ True,  True, False, False],\r\n       [False, False,  True, False],\r\n       [False, False, False,  True]])\r\n```\r\n\r\nThe output matrix is again, as expected, the outputs and inputs to the linear models are separable and independent of each other.\r\n\r\nIf however, I nest these compound 

In [156]:
index = 19

p_aware = exclusively_path_aware[index]
normal = how_they_failed[index]

print(p_aware["problem_statement"])
print("Ground Truth Patch File:", p_aware["patch_file"])
print("=================================")
print("Path Aware Retrieved Files:", p_aware["retrieved_files"][:10])
print("Normal Retrieved Files:", normal["retrieved_files"][:10])

Error creating AxisGrid with non-default axis class
<!--To help us understand and resolve your issue, please fill out the form to the best of your ability.-->
<!--You can feel free to delete the sections that do not apply.-->

### Bug report

**Bug summary**

Creating `AxesGrid` using cartopy `GeoAxes` as `axis_class` raises `TypeError: 'method' object is not subscriptable`. Seems to be due to different behaviour of `axis` attr. for `mpl_toolkits.axes_grid1.mpl_axes.Axes` and other axes instances (like `GeoAxes`) where `axis` is only a callable. The error is raised in method `mpl_toolkits.axes_grid1.axes_grid._tick_only` when trying to access keys from `axis` attr.

**Code for reproduction**

<!--A minimum code snippet required to reproduce the bug.
Please make sure to minimize the number of dependencies required, and provide
any necessary plotted data.
Avoid using threads, as Matplotlib is (explicitly) not thread-safe.-->

```python
import matplotlib.pyplot as plt
from cartopy.crs imp

In [157]:
# index 3, 5, 8, 16, 19, 27

In [199]:
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from ragent.chroma.chroma_store import get_splitter, load_or_build_vector_store

DATA_DIR = Path("./swebench_data")
REPO_ROOT = Path("./repo_data")
INCLUDE_PATH_IN_CHUNK = False

if INCLUDE_PATH_IN_CHUNK:
    CHROMA_ROOT = Path("chroma_data_test_path_aware")
else:
    CHROMA_ROOT = Path("chroma_data_test_nopath_aware")

EMBED_MODEL = "nomic-ai/nomic-embed-text-v1"
LLM_MODEL = "gpt-oss:120b"
EMBED_MODEL_KWARGS = {"device": "cuda", "trust_remote_code": True}
ENCODE_KWARGS = {"normalize_embeddings": False}


def make_embed_model(
    model_name: str = EMBED_MODEL,
    model_kwargs: dict = EMBED_MODEL_KWARGS,
    encode_kwargs: dict = ENCODE_KWARGS,
):
    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs,
        show_progress=True,
    )


In [200]:
analysis_data = exclusively_path_aware[19]

In [201]:
embed_model = make_embed_model()

<All keys matched successfully>


In [202]:
splitter = get_splitter("code")

[_get_splitter] using code splitter


In [203]:
analysis_data

{'swe_data_index': 142,
 'problem_statement': 'Error creating AxisGrid with non-default axis class\n<!--To help us understand and resolve your issue, please fill out the form to the best of your ability.-->\r\n<!--You can feel free to delete the sections that do not apply.-->\r\n\r\n### Bug report\r\n\r\n**Bug summary**\r\n\r\nCreating `AxesGrid` using cartopy `GeoAxes` as `axis_class` raises `TypeError: \'method\' object is not subscriptable`. Seems to be due to different behaviour of `axis` attr. for `mpl_toolkits.axes_grid1.mpl_axes.Axes` and other axes instances (like `GeoAxes`) where `axis` is only a callable. The error is raised in method `mpl_toolkits.axes_grid1.axes_grid._tick_only` when trying to access keys from `axis` attr.\r\n\r\n**Code for reproduction**\r\n\r\n<!--A minimum code snippet required to reproduce the bug.\r\nPlease make sure to minimize the number of dependencies required, and provide\r\nany necessary plotted data.\r\nAvoid using threads, as Matplotlib is (exp

In [204]:
from datasets import load_dataset
swebench = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")

swe_index = analysis_data["swe_data_index"]
swebench_entry = swebench[swe_index]

In [205]:
swebench_entry

{'repo': 'matplotlib/matplotlib',
 'instance_id': 'matplotlib__matplotlib-26020',
 'base_commit': 'f6a781f77f5ddf1204c60ca7c544809407d4a807',
 'patch': 'diff --git a/lib/mpl_toolkits/axes_grid1/axes_grid.py b/lib/mpl_toolkits/axes_grid1/axes_grid.py\n--- a/lib/mpl_toolkits/axes_grid1/axes_grid.py\n+++ b/lib/mpl_toolkits/axes_grid1/axes_grid.py\n@@ -1,5 +1,6 @@\n from numbers import Number\n import functools\n+from types import MethodType\n \n import numpy as np\n \n@@ -7,14 +8,20 @@\n from matplotlib.gridspec import SubplotSpec\n \n from .axes_divider import Size, SubplotDivider, Divider\n-from .mpl_axes import Axes\n+from .mpl_axes import Axes, SimpleAxisArtist\n \n \n def _tick_only(ax, bottom_on, left_on):\n     bottom_off = not bottom_on\n     left_off = not left_on\n-    ax.axis["bottom"].toggle(ticklabels=bottom_off, label=bottom_off)\n-    ax.axis["left"].toggle(ticklabels=left_off, label=left_off)\n+    if isinstance(ax.axis, MethodType):\n+        bottom = SimpleAxisArtist(ax.

In [206]:
vector_store = load_or_build_vector_store(
        swebench_entry,
        swe_index,
        embed_model,
        splitter,
        data_dir=DATA_DIR,
        chroma_root=CHROMA_ROOT,
        repo_root=REPO_ROOT,
        include_path_in_chunk=INCLUDE_PATH_IN_CHUNK,
    )

[load_or_build_vector_store] loading existing Chroma: chroma_data_test_nopath_aware/chroma_repo_index_swe_data142


In [207]:
def get_retriever_from_store(store, k: int = 1000):
    return store.as_retriever(search_kwargs={"k": k})

In [208]:
retriever = get_retriever_from_store(vector_store)

problem_statement = swebench_entry.get("problem_statement", "")
patch = swebench_entry.get("patch", "")

retrieved_docs = retriever.get_relevant_documents(problem_statement)
retrieved_paths = []
paths = set()
retrieved_chunks = []
for i, d in enumerate(retrieved_docs):
    paths.add(d.metadata.get("relative_path", ""))
    if d.metadata.get("relative_path", "") == analysis_data["patch_file"]:
        print(f"Found the actual patch in retrieved docs! Index: {i}")
        retrieved_chunks.append(d)
        # print(d)
    # print(d)
    # retrieved_paths.append(d.metadata.get("relative_path", ""))
    # print("=="*20)

Batches: 100%|██████████| 1/1 [00:00<00:00, 10.02it/s]

Found the actual patch in retrieved docs! Index: 37
Found the actual patch in retrieved docs! Index: 98
Found the actual patch in retrieved docs! Index: 427
Found the actual patch in retrieved docs! Index: 550
Found the actual patch in retrieved docs! Index: 989
Found the actual patch in retrieved docs! Index: 991


In [209]:
analysis_data

{'swe_data_index': 142,
 'problem_statement': 'Error creating AxisGrid with non-default axis class\n<!--To help us understand and resolve your issue, please fill out the form to the best of your ability.-->\r\n<!--You can feel free to delete the sections that do not apply.-->\r\n\r\n### Bug report\r\n\r\n**Bug summary**\r\n\r\nCreating `AxesGrid` using cartopy `GeoAxes` as `axis_class` raises `TypeError: \'method\' object is not subscriptable`. Seems to be due to different behaviour of `axis` attr. for `mpl_toolkits.axes_grid1.mpl_axes.Axes` and other axes instances (like `GeoAxes`) where `axis` is only a callable. The error is raised in method `mpl_toolkits.axes_grid1.axes_grid._tick_only` when trying to access keys from `axis` attr.\r\n\r\n**Code for reproduction**\r\n\r\n<!--A minimum code snippet required to reproduce the bug.\r\nPlease make sure to minimize the number of dependencies required, and provide\r\nany necessary plotted data.\r\nAvoid using threads, as Matplotlib is (exp

In [212]:
print(retrieved_chunks[1])

page_content='from numbers import Number
import functools

import numpy as np

from matplotlib import _api, cbook
from matplotlib.gridspec import SubplotSpec

from .axes_divider import Size, SubplotDivider, Divider
from .mpl_axes import Axes


def _tick_only(ax, bottom_on, left_on):
    bottom_off = not bottom_on
    left_off = not left_on
    ax.axis["bottom"].toggle(ticklabels=bottom_off, label=bottom_off)
    ax.axis["left"].toggle(ticklabels=left_off, label=left_off)


class CbarAxesBase:
    def __init__(self, *args, orientation, **kwargs):
        self.orientation = orientation
        super().__init__(*args, **kwargs)

    def colorbar(self, mappable, **kwargs):
        return self.figure.colorbar(
            mappable, cax=self, location=self.orientation, **kwargs)

    @_api.deprecated("3.8", alternative="ax.tick_params and colorbar.set_label")
    def toggle_label(self, b):
        axis = self.axis[self.orientation]
        axis.toggle(ticklabels=b, label=b)


_cbaraxes_class